# Mask Dataset Audit — CPU demo
Run both cells. Two deliberately planted errors are expected: an unknown label and a train/test duplicate. All images are synthetic. No upload or GPU required.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/B-jack7/mask-dataset-audit.git@main'])


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from PIL import Image
from mask_dataset_audit import audit
from mask_dataset_audit.report import render_html

with TemporaryDirectory() as directory:
    root = Path(directory)
    for split in ("train", "test"):
        (root / split / "images").mkdir(parents=True)
        (root / split / "masks").mkdir()
        image = np.arange(64, dtype=np.uint8).reshape(8, 8)
        mask = np.ones((8, 8), dtype=np.uint8)
        if split == "test":
            mask[0, 0] = 9
        Image.fromarray(image).save(root / split / "images/a.png")
        Image.fromarray(mask).save(root / split / "masks/a.png")
    report = audit(root, labels=[0, 1], splits=["train", "test"])
    assert report["summary"]["errors"] == 2
    print(report["summary"])
    try:
        from IPython.display import HTML, display
    except ImportError:
        print("Checks passed. Install IPython or use the CLI to view HTML.")
    else:
        display(HTML(render_html(report)))
